In [ ]:
#@title Load Azure OpenAI
# %%capture
# !pip install llama-index -q
# !pip install llama-index-llms-azure-openai -q
# !pip install llama-index-embeddings-azure-openai -q
!pip install llama-index-core -q
!pip install llama-index-utils-workflow -q

In [ ]:

from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from google.colab import userdata
from llama_index.core import Settings

llm = AzureOpenAI(
    engine="gpt-4o",
    deployment_name=userdata.get("AZURE_MODEL_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
    temperature=0.0001,
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=userdata.get("AZURE_EMBEDDING_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
)

Settings.llm = llm
Settings.embed_model = embed_model

In [ ]:
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
    Context
)

In [ ]:
class MyWorkflow(Workflow):
    # declare a function as a step
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # do something here
        return StopEvent(result="Hello, world!")

In [ ]:
# instantiate the workflow
basic_workflow = MyWorkflow(timeout=10, verbose=False)
# run the workflow
result = await basic_workflow.run()
print(result)

Hello, world!


In [ ]:
async def main():
    w = MyWorkflow(timeout=10, verbose=False)
    result = await w.run()
    print(result)

import nest_asyncio
nest_asyncio.apply()
if __name__ == "__main__":
    import asyncio
    asyncio.run(main())

Hello, world!


In [ ]:
from llama_index.utils.workflow import draw_all_possible_flows
import os
os.makedirs("workflows", exist_ok=True)
draw_all_possible_flows(
    basic_workflow,
    filename="workflows/basic_workflow.html",
    notebook=True
)

<class 'NoneType'>
<class 'llama_index.core.workflow.events.StopEvent'>
workflows/basic_workflow.html


In [ ]:
def extract_html_content(filename):
    try:
        with open(filename, 'r') as file:
            html_content = file.read()
            html_content = f""" <div style="width: 50%; height: 200px; overflow: hidden;"> {html_content} </div>"""
            return html_content
    except Exception as e:
        raise Exception(f"Error reading file: {str(e)}")

html_content = extract_html_content("workflows/basic_workflow.html")
# display(HTML("workflows/basic_workflow.html"), )

In [ ]:
from llama_index.core.workflow import Event

class FirstEvent(Event):
    first_output: str

class SecondEvent(Event):
    second_output: str

In [ ]:
class MyWorkflow(Workflow):
    @step
    async def stdfgep_two(self, ev: StartEvent) -> SecondEvent:
        print(ev.first_input)
        return SecondEvent(second_output="Second step complete.")

    @step
    async def stedfgp_one(self, ev: FirstEvent) -> FirstEvent:
        print(ev.first_output)
        return FirstEvent(first_output="First step complete.")

    @step
    async def step_three(self, ev: SecondEvent) -> StopEvent:
        print(ev.second_output)
        return StopEvent(result="Workflow complete.")

In [ ]:
workflow = MyWorkflow(timeout=10, verbose=False)
result = await workflow.run(first_input="Start the workflow.")
print(result)

Start the workflow.
Second step complete.
Workflow complete.


In [ ]:
workflow = MyWorkflow(timeout=10, verbose=False)
result = await workflow.run(first_input="Start the workflow.")
print(result)

Start the workflow.
Second step complete.
Workflow complete.


In [ ]:
WORKFLOW_FILE = "workflows/custom_events.html"
draw_all_possible_flows(workflow, filename=WORKFLOW_FILE)

<class 'NoneType'>
<class '__main__.SecondEvent'>
<class '__main__.FirstEvent'>
<class 'llama_index.core.workflow.events.StopEvent'>
workflows/custom_events.html


In [ ]:
class LoopEvent(Event):
    loop_output: str

In [ ]:
import random
class MyWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> FirstEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return FirstEvent(first_output="First step complete.")

    @step
    async def step_two(self, ev: FirstEvent) -> SecondEvent:
        print(ev.first_output)
        return SecondEvent(second_output="Second step complete.")

    @step
    async def step_three(self, ev: SecondEvent) -> StopEvent:
        print(ev.second_output)
        return StopEvent(result="Workflow complete.")

In [ ]:
loop_workflow = MyWorkflow(timeout=10, verbose=False)
result = await loop_workflow.run(first_input="Start the workflow.")
print(result)

Good thing happened
First step complete.
Second step complete.
Workflow complete.


In [ ]:
WORKFLOW_FILE = "workflows/loop_events.html"
draw_all_possible_flows(loop_workflow, filename=WORKFLOW_FILE)
html_content = extract_html_content(WORKFLOW_FILE)
# display(HTML(html_content), metadata=dict(isolated=True))

<class 'NoneType'>
<class '__main__.FirstEvent'>
<class '__main__.LoopEvent'>
<class 'llama_index.core.workflow.events.StopEvent'>
<class '__main__.SecondEvent'>
workflows/loop_events.html


In [ ]:
class BranchA1Event(Event):
    payload: str

class BranchA2Event(Event):
    payload: str

class BranchB1Event(Event):
    payload: str

class BranchB2Event(Event):
    payload: str

In [ ]:
class BranchWorkflow(Workflow):
    @step
    async def start(self, ev: StartEvent) -> BranchA1Event | BranchB1Event:
        if random.randint(0, 1) == 0:
            print("Go to branch A")
            return BranchA1Event(payload="Branch A")
        else:
            print("Go to branch B")
            return BranchB1Event(payload="Branch B")

    @step
    async def step_a1(self, ev: BranchA1Event) -> BranchA2Event:
        print(ev.payload)
        return BranchA2Event(payload=ev.payload)

    @step
    async def step_b1(self, ev: BranchB1Event) -> BranchB2Event:
        print(ev.payload)
        return BranchB2Event(payload=ev.payload)

    @step
    async def step_a2(self, ev: BranchA2Event) -> StopEvent:
        print(ev.payload)
        return StopEvent(result="Branch A complete.")

    @step
    async def step_b2(self, ev: BranchB2Event) -> StopEvent:
        print(ev.payload)
        return StopEvent(result="Branch B complete.")

In [ ]:
WORKFLOW_FILE = "workflows/branching.html"
draw_all_possible_flows(BranchWorkflow, filename=WORKFLOW_FILE)
html_content = extract_html_content(WORKFLOW_FILE)

<class 'NoneType'>
<class '__main__.BranchA1Event'>
<class '__main__.BranchB1Event'>
<class '__main__.BranchA2Event'>
<class 'llama_index.core.workflow.events.StopEvent'>
<class '__main__.BranchB2Event'>
<class 'llama_index.core.workflow.events.StopEvent'>
workflows/branching.html


In [ ]:
# display(HTML(html_content), metadata=dict(isolated=True))

In [ ]:
import asyncio

class StepTwoEvent(Event):
    query: str

class ParallelFlow(Workflow):
    @step
    async def start(self, ctx: Context, ev: StartEvent) -> StepTwoEvent:
        print("Starting parallel workflow")
        print(ev.anything)
        ctx.send_event(StepTwoEvent(query="Query 1"))
        ctx.send_event(StepTwoEvent(query="Query 2"))
        ctx.send_event(StepTwoEvent(query="Query 3"))

    @step(num_workers=4)
    async def step_two(self, ctx: Context, ev: StepTwoEvent) -> StopEvent:
        print("Running slow query ", ev.query)
        await asyncio.sleep(random.randint(1, 5))

        return StopEvent(result=ev.query)

In [ ]:
parallel_workflow = ParallelFlow(timeout=10, verbose=False)
result = await parallel_workflow.run(anything="Start the workflow.")
print(result)

Starting parallel workflow
Start the workflow.
Running slow query  Query 1
Running slow query  Query 2
Running slow query  Query 3
Query 2


In [ ]:
class StepThreeEvent(Event):
    result: str

class ConcurrentFlow(Workflow):
    @step
    async def start(self, ctx: Context, ev: StartEvent) -> StepTwoEvent:
        print("Starting concurrent workflow")
        print(ev.message)
        ctx.send_event(StepTwoEvent(query="Query 1"))
        ctx.send_event(StepTwoEvent(query="Query 2"))
        ctx.send_event(StepTwoEvent(query="Query 3"))

    @step(num_workers=4)
    async def step_two(self, ctx: Context, ev: StepTwoEvent) -> StepThreeEvent:
        print("Running query ", ev.query)
        await asyncio.sleep(random.randint(1, 5))
        return StepThreeEvent(result=ev.query)

    @step
    async def step_three(self, ctx: Context, ev: StepThreeEvent) -> StopEvent:
        # wait until we receive 3 events
        result = ctx.collect_events(ev, [StepThreeEvent] * 3)
        if result is None:
            print("Not all events received yet.")
            return None

        # do something with all 3 results together
        print(result)
        return StopEvent(result="Done")

In [ ]:
w = ConcurrentFlow(timeout=10, verbose=False)
result = await w.run(message="Start the workflow.")
# the result doesnt has the same executed order.
print(result)

Starting concurrent workflow
Start the workflow.
Running query  Query 1
Running query  Query 2
Running query  Query 3
Not all events received yet.
Not all events received yet.
[StepThreeEvent(result='Query 1'), StepThreeEvent(result='Query 2'), StepThreeEvent(result='Query 3')]
Done


In [ ]:
class StepAEvent(Event):
    query: str

class StepACompleteEvent(Event):
    result: str

class StepBEvent(Event):
    query: str

class StepBCompleteEvent(Event):
    result: str

class StepCEvent(Event):
    query: str

class StepCCompleteEvent(Event):
    result: str

In [ ]:
class ConcurrentFlow(Workflow):
    @step
    async def start(
        self, ctx: Context, ev: StartEvent
    ) -> StepAEvent | StepBEvent | StepCEvent:
        ctx.send_event(StepAEvent(query="Query 1"))
        ctx.send_event(StepBEvent(query="Query 2"))
        ctx.send_event(StepCEvent(query="Query 3"))

    @step
    async def step_a(self, ctx: Context, ev: StepAEvent) -> StepACompleteEvent:
        print("Doing something A-ish")
        return StepACompleteEvent(result=ev.query)

    @step
    async def step_b(self, ctx: Context, ev: StepBEvent) -> StepBCompleteEvent:
        print("Doing something B-ish")
        return StepBCompleteEvent(result=ev.query)

    @step
    async def step_c(self, ctx: Context, ev: StepCEvent) -> StepCCompleteEvent:
        print("Doing something C-ish")
        return StepCCompleteEvent(result=ev.query)

    @step
    async def step_three(
        self,
        ctx: Context,
        ev: StepACompleteEvent | StepBCompleteEvent | StepCCompleteEvent,
    ) -> StopEvent:
        print("Received event ", ev.result)

        # wait until we receive 3 events
        events = ctx.collect_events(
            ev,
            [StepCCompleteEvent, StepACompleteEvent, StepBCompleteEvent],
        )
        if (events is None):
            return None

        # do something with all 3 results together
        print("All events received: ", events)
        return StopEvent(result="Done")

In [ ]:
w = ConcurrentFlow(timeout=10, verbose=False)
result = await w.run(message="Start the workflow.")
print(result)

Doing something A-ish
Doing something B-ish
Doing something C-ish
Received event  Query 1
Received event  Query 2
Received event  Query 3
All events received:  [StepCCompleteEvent(result='Query 3'), StepACompleteEvent(result='Query 1'), StepBCompleteEvent(result='Query 2')]
Done


In [ ]:
WORKFLOW_FILE = "workflows/concurrent_different_events.html"
draw_all_possible_flows(w, filename=WORKFLOW_FILE)
html_content = extract_html_content(WORKFLOW_FILE)

<class 'NoneType'>
<class '__main__.StepAEvent'>
<class '__main__.StepBEvent'>
<class '__main__.StepCEvent'>
<class '__main__.StepACompleteEvent'>
<class '__main__.StepBCompleteEvent'>
<class '__main__.StepCCompleteEvent'>
<class 'llama_index.core.workflow.events.StopEvent'>
workflows/concurrent_different_events.html


# RAG

In [ ]:
from llama_parse import LlamaParse

In [ ]:
userdata.get("AZURE_BASE_URL")

'https://gps-cws-eus-aoi.openai.azure.com/'

In [ ]:
userdata.get("AZURE_API_KEY")

'8c459b9b6b8e4351bb2a819680b891c9'

In [ ]:
parser = LlamaParse(
    api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
    # use_vendor_multimodal_model=True,
    azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
    azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
    azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
    azure_openai_key=userdata.get("AZURE_API_KEY"),
    result_type="markdown",
    content_guideline_instruction="This is a resume, gather related facts together and format it as bullet points with headers"
)

In [ ]:
!gdown 1s9izRjdg9NoQzrq2U7l9XumVqwL3Q4xq

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1s9izRjdg9NoQzrq2U7l9XumVqwL3Q4xq

but Gdown can't. Please check connections and permissions.


In [ ]:
documents = parser.load_data(file_path="fake_resume.pdf")

Started parsing the file under job_id 09a72be7-ae5d-4a2f-97f3-3f8f634fbb47


In [ ]:
print(documents[2].text)

# Projects

# EcoTrack | GitHub

- Built full-stack application for tracking carbon footprint using React, Node.js, and MongoDB
- Implemented machine learning algorithm for providing personalized sustainability recommendations
- Featured in TechCrunch's "Top 10 Environmental Impact Apps of 2023"

# ChatFlow | Demo

- Developed real-time chat application using WebSocket protocol and React
- Implemented end-to-end encryption and message persistence
- Serves 5000+ monthly active users

# Certifications

- AWS Certified Solutions Architect (2023)
- Google Cloud Professional Developer (2022)
- MongoDB Certified Developer (2021)

# Languages

- English (Native)
- Mandarin Chinese (Fluent)
- Spanish (Intermediate)

# Interests

- Open source contribution
- Tech blogging (15K+ Medium followers)
- Hackathon mentoring
- Rock climbing


In [ ]:
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.core import VectorStoreIndex

In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model
)

In [ ]:
query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)
response = query_engine.query("What is this person's name and what was their most recent job?")
print(response)

Nicolas CARPi is the person's name, and their most recent job was at Institut Curie UMR144/CNRS in Paris, France.


In [ ]:
storage_dir = "./storage"

index.storage_context.persist(persist_dir=storage_dir)

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
# Check if the index is stored on disk
if os.path.exists(storage_dir):
    # Load the index from disk
    storage_context = StorageContext.from_defaults(persist_dir=storage_dir)
    restored_index = load_index_from_storage(storage_context)
else:
    print("Index not found on disk.")

In [ ]:
response = restored_index.as_query_engine().query("What is this person's name and what was their most recent job?")
print(response)

Nicolas CARPi is the person's name, and their most recent job was at Institut Curie UMR144/CNRS in Paris, France.


In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import FunctionCallingAgent
def query_resume(q: str) -> str:
    """Answers questions about a specific resume."""
    # we're using the query engine we already created above
    response = query_engine.query(f"This is a question about the specific resume we have in our database: {q}")
    return response.response

In [ ]:
resume_tool = FunctionTool.from_defaults(fn=query_resume)

In [ ]:
agent = FunctionCallingAgent.from_tools(
    tools=[resume_tool],
    llm=llm,
    verbose=True
)

In [ ]:
response = agent.chat("How many years of experience does the applicant have?")
print(response)

> Running step 5da80112-bd65-4c84-a48a-4773c7dcef65. Step input: How many years of experience does the applicant have?
Added user message to memory: How many years of experience does the applicant have?
=== Calling Function ===
Calling function: query_resume with args: {"q": "How many years of experience does the applicant have?"}
=== Function Output ===
The provided context does not contain information about the applicant's years of experience.
> Running step 7f135fe8-a4ea-45e0-9201-0b49244ef199. Step input: None
=== LLM Response ===
The resume does not contain information about the applicant's years of experience.
The resume does not contain information about the applicant's years of experience.


In [ ]:
from llama_index.core.workflow import (
    StartEvent,
    StopEvent,
    Workflow,
    step,
    Event,
    Context
)

In [ ]:
class QueryEvent(Event):
    query: str

In [ ]:
class RAGWorkflow(Workflow):
    storage_dir = "./storage"
    llm: AzureOpenAI
    query_engine: VectorStoreIndex

    # the first step will be setup
    @step
    async def set_up(self, ctx: Context, ev: StartEvent) -> QueryEvent:

        if not ev.resume_file:
            raise ValueError("No resume file provided")

        # define an LLM to work with
        self.llm = llm

        # ingest the data and set up the query engine
        if os.path.exists(self.storage_dir):
            # you've already ingested your documents
            storage_context = StorageContext.from_defaults(persist_dir=self.storage_dir)
            index = load_index_from_storage(storage_context)
        else:
            # parse and load your documents
            documents = LlamaParse(
                api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
                # use_vendor_multimodal_model=True,
                azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
                azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
                azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
                azure_openai_key=userdata.get("AZURE_API_KEY"),
                result_type="markdown",
                content_guideline_instruction="This is a resume, gather related facts together and format it as bullet points with headers"
            ).load_data(ev.resume_file)
            # embed and index the documents
            index = VectorStoreIndex.from_documents(
                documents,
                embed_model=embed_model
            )
            index.storage_context.persist(persist_dir=self.storage_dir)

        # either way, create a query engine
        self.query_engine = index.as_query_engine(llm=self.llm, similarity_top_k=5)

        # now fire off a query event to trigger the next step
        return QueryEvent(query=ev.query)

    # the second step will be to ask a question and return a result immediately
    @step
    async def ask_question(self, ctx: Context, ev: QueryEvent) -> StopEvent:
        response = self.query_engine.query(f"This is a question about the specific resume we have in our database: {ev.query}")
        return StopEvent(result=response.response)

In [ ]:
w = RAGWorkflow(timeout=120, verbose=False)
result = await w.run(
    resume_file="fake_resume.pdf",
    query="Where is the first place the applicant worked?"
)
print(result)

The first place the applicant worked is the Institut Curie UMR144/CNRS in Paris, France.


In [ ]:
WORKFLOW_FILE = "workflows/rag_workflow.html"
draw_all_possible_flows(w, filename=WORKFLOW_FILE)
html_content = extract_html_content(WORKFLOW_FILE)
display(HTML(html_content), metadata=dict(isolated=True))

<class 'NoneType'>
<class 'llama_index.core.workflow.events.StopEvent'>
<class '__main__.QueryEvent'>
workflows/rag_workflow.html


In [ ]:
!gdown 1nRj8FNyI2YQ_tBnbbj6Vy4hBUs42fsL9

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1nRj8FNyI2YQ_tBnbbj6Vy4hBUs42fsL9

but Gdown can't. Please check connections and permissions.


In [ ]:
result = parser.load_data("fake_application_form.pdf")[0]

Started parsing the file under job_id fe75ba9f-8e94-4ee7-af8a-1c437a5c3b65


In [ ]:
print(result.text)

# Big Tech Co. Job Application Form

# Position: Senior Web Developer C3

Thanks for applying to Big Tech Co.! We are humbled that you would consider working here.

Please fill in the following form to help us get started.

|First Name|Last Name|
|---|---|
|Email|Phone|
|Linkedin| |
|Project Portfolio| |
|Degree|Graduation Date|
|Current Job title|Current Employer|
|Technical Skills| |
|Describe why you’re a good fit for this position|Describe why you’re a good fit for this position|
|Do you have 5 years of experience in React?|Do you have 5 years of experience in React?|


In [ ]:
raw_json = llm.complete(
    f"""
    This is a parsed form.
    Convert it into a JSON object containing only the list
    of fields to be filled in, in the form {{ fields: [...] }}.
    <form>{result.text}</form>.
    Return JSON ONLY, no markdown."""
)

In [ ]:
raw_json.text

'{\n  "fields": [\n    "First Name",\n    "Last Name",\n    "Email",\n    "Phone",\n    "Linkedin",\n    "Project Portfolio",\n    "Degree",\n    "Graduation Date",\n    "Current Job title",\n    "Current Employer",\n    "Technical Skills",\n    "Describe why you’re a good fit for this position",\n    "Do you have 5 years of experience in React?"\n  ]\n}'

In [ ]:
import json
fields = json.loads(raw_json.text)["fields"]

for field in fields:
    print(field)

First Name
Last Name
Email
Phone
Linkedin
Project Portfolio
Degree
Graduation Date
Current Job title
Current Employer
Technical Skills
Describe why you’re a good fit for this position
Do you have 5 years of experience in React?


In [ ]:
class ParseFormEvent(Event):
    application_form: str

class QueryEvent(Event):
    query: str

In [ ]:
class RAGWorkflow(Workflow):

    storage_dir = "./storage"
    llm: AzureOpenAI
    query_engine: VectorStoreIndex

    @step
    async def set_up(self, ctx: Context, ev: StartEvent) -> ParseFormEvent:

        if not ev.resume_file:
            raise ValueError("No resume file provided")

        if not ev.application_form:
            raise ValueError("No application form provided")

        # define the LLM to work with
        self.llm = llm

        # ingest the data and set up the query engine
        if os.path.exists(self.storage_dir):
            # you've already ingested the resume document
            storage_context = StorageContext.from_defaults(persist_dir=self.storage_dir)
            index = load_index_from_storage(storage_context)
        else:
            # parse and load the resume document
            documents = LlamaParse(
                api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
                # use_vendor_multimodal_model=True,
                azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
                azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
                azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
                azure_openai_key=userdata.get("AZURE_API_KEY"),
                result_type="markdown",
                content_guideline_instruction="This is a resume, gather related facts together and format it as bullet points with headers"
            ).load_data(ev.resume_file)
            # embed and index the documents
            index = VectorStoreIndex.from_documents(
                documents,
                embed_model=embed_model
            )
            index.storage_context.persist(persist_dir=self.storage_dir)

        # create a query engine
        self.query_engine = index.as_query_engine(llm=self.llm, similarity_top_k=5)

        # you no longer need a query to be passed in,
        # you'll be generating the queries instead
        # let's pass the application form to a new step to parse it
        return ParseFormEvent(application_form=ev.application_form)

    @step
    async def parse_form(self, ctx: Context, ev: ParseFormEvent) -> QueryEvent:
        parser = LlamaParse(
            api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
            # use_vendor_multimodal_model=True,
            azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
            azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
            azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
            azure_openai_key=userdata.get("AZURE_API_KEY"),
            result_type="markdown",
            content_guideline_instruction="This is a job application form. Create a list of all the fields that need to be filled in.",
            formatting_instruction="Return a bulleted list of the fields ONLY."
        )

        # get the LLM to convert the parsed form into JSON
        result = parser.load_data(ev.application_form)[0]
        raw_json = self.llm.complete(
            f"""
            This is a parsed form.
            Convert it into a JSON object containing only the list
            of fields to be filled in, in the form {{ fields: [...] }}.
            <form>{result.text}</form>.
            Return JSON ONLY, no markdown.
            """)
        fields = json.loads(raw_json.text)["fields"]

        for field in fields:
            print(field)
        return StopEvent(result="Dummy event")

    # will be edited in the next section
    @step
    async def ask_question(self, ctx: Context, ev: QueryEvent) -> StopEvent:
        response = self.query_engine.query(f"This is a question about the specific resume we have in our database: {ev.query}")
        return StopEvent(result=response.response)

In [ ]:
w = RAGWorkflow(timeout=60, verbose=False)
result = await w.run(
    resume_file="fake_resume.pdf",
    application_form="fake_application_form.pdf"
)

Started parsing the file under job_id 4f0c6418-9d4f-4887-8932-163fcc0e1e69
First Name
Last Name
Email
Phone
Linkedin
Project Portfolio
Degree
Graduation Date
Current Job title
Current Employer
Technical Skills
Describe why you’re a good fit for this position
Do you have 5 years of experience in React?


In [ ]:
result

'Dummy event'

In [ ]:
class ParseFormEvent(Event):
    application_form: str

class QueryEvent(Event):
    query: str
    field: str

# new!
class ResponseEvent(Event):
    response: str

In [ ]:
class RAGWorkflow(Workflow):

    storage_dir = "./storage"
    llm: AzureOpenAI
    query_engine: VectorStoreIndex

    @step
    async def set_up(self, ctx: Context, ev: StartEvent) -> ParseFormEvent:

        if not ev.resume_file:
            raise ValueError("No resume file provided")

        if not ev.application_form:
            raise ValueError("No application form provided")

        # define the LLM to work with
        self.llm = llm

        # ingest the data and set up the query engine
        if os.path.exists(self.storage_dir):
            # you've already ingested the resume document
            storage_context = StorageContext.from_defaults(persist_dir=self.storage_dir)
            index = load_index_from_storage(storage_context)
        else:
            # parse and load the resume document
            documents = LlamaParse(
                api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
                # use_vendor_multimodal_model=True,
                azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
                azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
                azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
                azure_openai_key=userdata.get("AZURE_API_KEY"),
                result_type="markdown",
                content_guideline_instruction="This is a resume, gather related facts together and format it as bullet points with headers"
            ).load_data(ev.resume_file)
            # embed and index the documents
            index = VectorStoreIndex.from_documents(
                documents,
                embed_model=embed_model
            )
            index.storage_context.persist(persist_dir=self.storage_dir)

        # create a query engine
        self.query_engine = index.as_query_engine(llm=self.llm, similarity_top_k=5)

        # you no longer need a query to be passed in,
        # you'll be generating the queries instead
        # let's pass the application form to a new step to parse it
        return ParseFormEvent(application_form=ev.application_form)

    @step
    async def parse_form(self, ctx: Context, ev: ParseFormEvent) -> QueryEvent:
        parser = LlamaParse(
            api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
            # use_vendor_multimodal_model=True,
            azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
            azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
            azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
            azure_openai_key=userdata.get("AZURE_API_KEY"),
            result_type="markdown",
            content_guideline_instruction="This is a job application form. Create a list of all the fields that need to be filled in.",
            formatting_instruction="Return a bulleted list of the fields ONLY."
        )

        # get the LLM to convert the parsed form into JSON
        result = parser.load_data(ev.application_form)[0]
        raw_json = self.llm.complete(
            f"""
            This is a parsed form.
            Convert it into a JSON object containing only the list
            of fields to be filled in, in the form {{ fields: [...] }}.
            <form>{result.text}</form>.
            Return JSON ONLY, no markdown.
            """)
        fields = json.loads(raw_json.text)["fields"]

        # new!
        # generate one query for each of the fields, and fire them off
        for field in fields:
            ctx.send_event(QueryEvent(
                field=field,
                query=f"How would you answer this question about the candidate? {field}"
            ))

        # store the number of fields so we know how many to wait for later
        await ctx.set("total_fields", len(fields))
        return

    @step
    async def ask_question(self, ctx: Context, ev: QueryEvent) -> ResponseEvent:
        response = self.query_engine.query(f"This is a question about the specific resume we have in our database: {ev.query}")
        return ResponseEvent(field=ev.field, response=response.response)

    # new!
    @step
    async def fill_in_application(self, ctx: Context, ev: ResponseEvent) -> StopEvent:
        # get the total number of fields to wait for
        total_fields = await ctx.get("total_fields")

        responses = ctx.collect_events(ev, [ResponseEvent] * total_fields)
        if responses is None:
            return None # do nothing if there's nothing to do yet

        # we've got all the responses!
        responseList = "\n".join("Field: " + r.field + "\n" + "Response: " + r.response for r in responses)

        result = self.llm.complete(f"""
            You are given a list of fields in an application form and responses to
            questions about those fields from a resume. Combine the two into a list of
            fields and succinct, factual answers to fill in those fields.

            <responses>
            {responseList}
            </responses>
        """)
        return StopEvent(result=result)

In [ ]:
w = RAGWorkflow(timeout=120, verbose=False)
result = await w.run(
    resume_file="fake_resume.pdf",
    application_form="fake_application_form.pdf"
)
print(result)

Started parsing the file under job_id eb50b12f-eb8e-430f-a155-c6f3a0260cd2
<combined>
Field: First Name
Response: Nicolas

Field: Last Name
Response: CARPi

Field: Email
Response: Not provided

Field: Phone
Response: Not provided

Field: Linkedin
Response: The candidate has contributed to the development of eLabFTW, an open source electronic laboratory notebook for research labs. They have experience in creating software that helps researchers track experiments and manage lab assets. Additionally, they have been involved in ensuring the software includes features like timestamping for legal proof and equipment scheduling. Their work has been published and is accessible through various online resources and documentation.

Field: Project Portfolio
Response: The candidate has worked on eLabFTW, a free and open source electronic laboratory notebook designed for researchers. This project allows researchers to track their experiments and manage lab assets such as antibodies, mice, siRNAs, an

In [ ]:
WORKFLOW_FILE = "workflows/form_parsing_workflow.html"
draw_all_possible_flows(w, filename=WORKFLOW_FILE)
html_content = extract_html_content(WORKFLOW_FILE)
display(HTML(html_content), metadata=dict(isolated=True))

<class 'NoneType'>
<class '__main__.ResponseEvent'>
<class 'llama_index.core.workflow.events.StopEvent'>
<class '__main__.QueryEvent'>
<class '__main__.ParseFormEvent'>
workflows/form_parsing_workflow.html


In [ ]:
from llama_index.core.workflow import InputRequiredEvent, HumanResponseEvent

In [ ]:
class ParseFormEvent(Event):
    application_form: str

class QueryEvent(Event):
    query: str
    field: str

class ResponseEvent(Event):
    response: str

# new!
class FeedbackEvent(Event):
    feedback: str

class GenerateQuestionsEvent(Event):
    pass

In [ ]:
class RAGWorkflow(Workflow):

    storage_dir = "./storage"
    llm: AzureOpenAI
    query_engine: VectorStoreIndex

    @step
    async def set_up(self, ctx: Context, ev: StartEvent) -> ParseFormEvent:

        if not ev.resume_file:
            raise ValueError("No resume file provided")

        if not ev.application_form:
            raise ValueError("No application form provided")

        # define the LLM to work with
        self.llm = llm

        # ingest the data and set up the query engine
        if os.path.exists(self.storage_dir):
            # you've already ingested the resume document
            storage_context = StorageContext.from_defaults(persist_dir=
                                                           self.storage_dir)
            index = load_index_from_storage(storage_context)
        else:
            # parse and load the resume document
            documents = LlamaParse(
                api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
                # use_vendor_multimodal_model=True,
                azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
                azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
                azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
                azure_openai_key=userdata.get("AZURE_API_KEY"),
                result_type="markdown",
                content_guideline_instruction="This is a resume, gather related facts together and format it as bullet points with headers"
            ).load_data(ev.resume_file)
            # embed and index the documents
            index = VectorStoreIndex.from_documents(
                documents,
                embed_model=embed_model
            )
            index.storage_context.persist(persist_dir=self.storage_dir)

        # create a query engine
        self.query_engine = index.as_query_engine(llm=self.llm, similarity_top_k=5)

        # you no longer need a query to be passed in,
        # you'll be generating the queries instead
        # let's pass the application form to a new step to parse it
        return ParseFormEvent(application_form=ev.application_form)

    # new - separated the form parsing from the question generation
    @step
    async def parse_form(self, ctx: Context, ev: ParseFormEvent) -> GenerateQuestionsEvent:
        parser = LlamaParse(
            api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
            # use_vendor_multimodal_model=True,
            azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
            azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
            azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
            azure_openai_key=userdata.get("AZURE_API_KEY"),
            result_type="markdown",
            content_guideline_instruction="This is a job application form. Create a list of all the fields that need to be filled in.",
            formatting_instruction="Return a bulleted list of the fields ONLY."
        )

        # get the LLM to convert the parsed form into JSON
        result = parser.load_data(ev.application_form)[0]
        raw_json = self.llm.complete(
            f"This is a parsed form. Convert it into a JSON object containing only the list of fields to be filled in, in the form {{ fields: [...] }}. <form>{result.text}</form>. Return JSON ONLY, no markdown.")
        fields = json.loads(raw_json.text)["fields"]

        await ctx.set("fields_to_fill", fields)

        return GenerateQuestionsEvent()

    # new - this step can get triggered either by GenerateQuestionsEvent or a FeedbackEvent
    @step
    async def generate_questions(self, ctx: Context, ev: GenerateQuestionsEvent | FeedbackEvent) -> QueryEvent:

        # get the list of fields to fill in
        fields = await ctx.get("fields_to_fill")

        # generate one query for each of the fields, and fire them off
        for field in fields:
            question = f"How would you answer this question about the candidate? <field>{field}</field>"
            ctx.send_event(QueryEvent(
                field=field,
                query=question
            ))

        # store the number of fields so we know how many to wait for later
        await ctx.set("total_fields", len(fields))
        return

    @step
    async def ask_question(self, ctx: Context, ev: QueryEvent) -> ResponseEvent:
        response = self.query_engine.query(f"This is a question about the specific resume we have in our database: {ev.query}")
        return ResponseEvent(field=ev.field, response=response.response)


    # new - we now emit an InputRequiredEvent
    @step
    async def fill_in_application(self, ctx: Context, ev: ResponseEvent) -> InputRequiredEvent:
        # get the total number of fields to wait for
        total_fields = await ctx.get("total_fields")

        responses = ctx.collect_events(ev, [ResponseEvent] * total_fields)
        if responses is None:
            return None # do nothing if there's nothing to do yet

        # we've got all the responses!
        responseList = "\n".join("Field: " + r.field + "\n" + "Response: " + r.response for r in responses)

        result = self.llm.complete(f"""
            You are given a list of fields in an application form and responses to
            questions about those fields from a resume. Combine the two into a list of
            fields and succinct, factual answers to fill in those fields.

            <responses>
            {responseList}
            </responses>
        """)

        # new! save the result for later
        await ctx.set("filled_form", str(result))

        # new! Let's get a human in the loop
        return InputRequiredEvent(
            prefix="How does this look? Give me any feedback you have on any of the answers.",
            result=result
        )

    # new! Accept the feedback.
    @step
    async def get_feedback(self, ctx: Context, ev: HumanResponseEvent) -> FeedbackEvent | StopEvent:

        result = self.llm.complete(f"""
            You have received some human feedback on the form-filling task you've done.
            Does everything look good, or is there more work to be done?
            <feedback>
            {ev.response}
            </feedback>
            If everything is fine, respond with just the word 'OKAY'.
            If there's any other feedback, respond with just the word 'FEEDBACK'.
        """)

        verdict = result.text.strip()

        print(f"LLM says the verdict was {verdict}")
        if (verdict == "OKAY"):
            return StopEvent(result=await ctx.get("filled_form"))
        else:
            return FeedbackEvent(feedback=ev.response)


In [ ]:
w = RAGWorkflow(timeout=600, verbose=False)
handler = w.run(
    resume_file="fake_resume.pdf",
    application_form="fake_application_form.pdf"
)

async for event in handler.stream_events():
    if isinstance(event, InputRequiredEvent):
        print("We've filled in your form! Here are the results:\n")
        print(event.result)
        # now ask for input from the keyboard
        response = input(event.prefix)
        handler.ctx.send_event(
            HumanResponseEvent(
                response=response
            )
        )

response = await handler
print("Agent complete! Here's your final result:")
print(str(response))
# prompts
# portfolio should be a URL
# This is fine

Started parsing the file under job_id efd822ef-6290-4cd6-a37e-0427e721f1b2
We've filled in your form! Here are the results:

Here is the combined list of fields with succinct, factual answers based on the provided responses:

1. **First Name**: Nicolas
2. **Last Name**: CARPi
3. **Email**: Not provided
4. **Phone**: Not provided
5. **LinkedIn**: Not provided
6. **Project Portfolio**: Worked on eLabFTW, a free and open source electronic laboratory notebook designed for researchers. This project involves tracking experiments, managing lab assets, and providing timestamping for legal proof in patent issues. The software also includes a scheduler for booking equipment.
7. **Degree**: Degree from Institut Curie UMR144/CNRS in Paris, France, and from the Institute of Biochemical Plant Physiology at Heinrich Heine University in Düsseldorf, Germany.
8. **Graduation Date**: Not provided
9. **Current Job Title**: Not provided
10. **Current Employer**: Institut Curie UMR144/CNRS
11. **Technical S

In [ ]:
class RAGWorkflow(Workflow):

    storage_dir = "./storage"
    llm: AzureOpenAI
    query_engine: VectorStoreIndex

    @step
    async def set_up(self, ctx: Context, ev: StartEvent) -> ParseFormEvent:

        if not ev.resume_file:
            raise ValueError("No resume file provided")

        if not ev.application_form:
            raise ValueError("No application form provided")

        # define the LLM to work with
        self.llm = llm

        # ingest the data and set up the query engine
        if os.path.exists(self.storage_dir):
            # you've already ingested the resume document
            storage_context = StorageContext.from_defaults(persist_dir=
                                                           self.storage_dir)
            index = load_index_from_storage(storage_context)
        else:
            # parse and load the resume document
            documents = LlamaParse(
                api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
                # use_vendor_multimodal_model=True,
                azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
                azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
                azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
                azure_openai_key=userdata.get("AZURE_API_KEY"),
                result_type="markdown",
                content_guideline_instruction="This is a resume, gather related facts together and format it as bullet points with headers"
            ).load_data(ev.resume_file)
            # embed and index the documents
            index = VectorStoreIndex.from_documents(
                documents,
                embed_model=embed_model
            )
            index.storage_context.persist(persist_dir=self.storage_dir)

        # create a query engine
        self.query_engine = index.as_query_engine(llm=self.llm, similarity_top_k=5)

        # let's pass the application form to a new step to parse it
        return ParseFormEvent(application_form=ev.application_form)

    # form parsing
    @step
    async def parse_form(self, ctx: Context, ev: ParseFormEvent) -> GenerateQuestionsEvent:
        parser = LlamaParse(
            api_key='llx-T2M93gdnXJ64adwRpl27s2U17S0cImnHKQVdSo1cou7nTIIt',
            # use_vendor_multimodal_model=True,
            azure_openai_endpoint=userdata.get("AZURE_BASE_URL"),
            azure_openai_deployment_name=userdata.get("AZURE_MODEL_NAME"),
            azure_openai_api_version=userdata.get("AZURE_API_VERSION"),
            azure_openai_key=userdata.get("AZURE_API_KEY"),
            result_type="markdown",
            content_guideline_instruction="This is a job application form. Create a list of all the fields that need to be filled in.",
            formatting_instruction="Return a bulleted list of the fields ONLY."
        )

        # get the LLM to convert the parsed form into JSON
        result = parser.load_data(ev.application_form)[0]
        raw_json = self.llm.complete(
            f"This is a parsed form. Convert it into a JSON object containing only the list of fields to be filled in, in the form {{ fields: [...] }}. <form>{result.text}</form>. Return JSON ONLY, no markdown.")
        fields = json.loads(raw_json.text)["fields"]

        await ctx.set("fields_to_fill", fields)

        return GenerateQuestionsEvent()

    # generate questions
    @step
    async def generate_questions(self, ctx: Context, ev: GenerateQuestionsEvent | FeedbackEvent) -> QueryEvent:

        # get the list of fields to fill in
        fields = await ctx.get("fields_to_fill")

        # generate one query for each of the fields, and fire them off
        for field in fields:
            question = f"How would you answer this question about the candidate? <field>{field}</field>"

            # new! Is there feedback? If so, add it to the query:
            if hasattr(ev,"feedback"):
                question += f"""
                    \nWe previously got feedback about how we answered the questions.
                    It might not be relevant to this particular field, but here it is:
                    <feedback>{ev.feedback}</feedback>
                """

            ctx.send_event(QueryEvent(
                field=field,
                query=question
            ))

        # store the number of fields so we know how many to wait for later
        await ctx.set("total_fields", len(fields))
        return

    @step
    async def ask_question(self, ctx: Context, ev: QueryEvent) -> ResponseEvent:
        response = self.query_engine.query(f"This is a question about the specific resume we have in our database: {ev.query}")
        return ResponseEvent(field=ev.field, response=response.response)


    # Get feedback from the human
    @step
    async def fill_in_application(self, ctx: Context, ev: ResponseEvent) -> InputRequiredEvent:
        # get the total number of fields to wait for
        total_fields = await ctx.get("total_fields")

        responses = ctx.collect_events(ev, [ResponseEvent] * total_fields)
        if responses is None:
            return None # do nothing if there's nothing to do yet

        # we've got all the responses!
        responseList = "\n".join("Field: " + r.field + "\n" + "Response: " + r.response for r in responses)

        result = self.llm.complete(f"""
            You are given a list of fields in an application form and responses to
            questions about those fields from a resume. Combine the two into a list of
            fields and succinct, factual answers to fill in those fields.

            <responses>
            {responseList}
            </responses>
        """)

        # save the result for later
        await ctx.set("filled_form", str(result))

        # Fire off the feedback request
        return InputRequiredEvent(
            prefix="How does this look? Give me any feedback you have on any of the answers.",
            result=result
        )

    # Accept the feedback when a HumanResponseEvent fires
    @step
    async def get_feedback(self, ctx: Context, ev: HumanResponseEvent) -> FeedbackEvent | StopEvent:

        result = self.llm.complete(f"""
            You have received some human feedback on the form-filling task you've done.
            Does everything look good, or is there more work to be done?
            <feedback>
            {ev.response}
            </feedback>
            If everything is fine, respond with just the word 'OKAY'.
            If there's any other feedback, respond with just the word 'FEEDBACK'.
        """)

        verdict = result.text.strip()

        print(f"LLM says the verdict was {verdict}")
        if (verdict == "OKAY"):
            return StopEvent(result=await ctx.get("filled_form"))
        else:
            return FeedbackEvent(feedback=ev.response)


In [ ]:
w = RAGWorkflow(timeout=600, verbose=False)
handler = w.run(
    resume_file="fake_resume.pdf",
    application_form="fake_application_form.pdf"
)

async for event in handler.stream_events():
    if isinstance(event, InputRequiredEvent):
        print("We've filled in your form! Here are the results:\n")
        print(event.result)
        # now ask for input from the keyboard
        response = input(event.prefix)
        handler.ctx.send_event(
            HumanResponseEvent(
                response=response
            )
        )

response = await handler
print("Agent complete! Here's your final result:")
print(str(response))
# Prompts
# Portfolio should be a URL
# That's great!

Started parsing the file under job_id 1f2cdebf-071d-461a-8d94-0d794c76436d
We've filled in your form! Here are the results:

1. **First Name**: Nicolas
2. **Last Name**: CARPi
3. **Email**: Not provided
4. **Phone**: Not provided
5. **LinkedIn**: Not provided
6. **Project Portfolio**: Worked on eLabFTW, a free and open source electronic laboratory notebook designed for researchers. This project involves tracking experiments, managing lab assets, and providing timestamping for legal proof in patent issues. The software also includes a scheduler for booking equipment.
7. **Degree**: Degree from Institut Curie UMR144/CNRS in Paris, France, and from the Institute of Biochemical Plant Physiology at Heinrich Heine University in Düsseldorf, Germany.
8. **Graduation Date**: Not provided
9. **Current Job Title**: Not provided
10. **Current Employer**: Institut Curie UMR144/CNRS
11. **Technical Skills**: Experience with eLabFTW, a free and open source electronic laboratory notebook designed for 